In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.drift_monitor;
CREATE VOLUME IF NOT EXISTS workspace.drift_monitor.bronze;

In [0]:
BRONZE_PATH = "/Volumes/workspace/drift_monitor/bronze/transactions/"
files = dbutils.fs.ls(BRONZE_PATH)
print(len(files))

In [0]:
bronze = spark.read.parquet(BRONZE_PATH)
bronze.printSchema()

In [0]:
print(bronze.count())

In [0]:
display(bronze.limit(5))

Reconciliation Checks

- **Reconciliation:** Spark on bronze (Databricks) matches DuckDB on the raw CSV exactly: 6,362,620 rows, identical fraud counts per type.

In [0]:
from pyspark.sql import functions as F

fraud_by_type = (
    bronze.groupBy("transaction_type")
    .agg(
        F.count("*").alias("total_txns"),
        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txns"),
    )
    .withColumn("fraud_rate_pct", F.round(100 * F.col("fraud_txns") / F.col("total_txns"), 3))
    .orderBy(F.desc("fraud_rate_pct"))
)
display(fraud_by_type)

Checking the overdrawned Transactions

In [0]:
from pyspark.sql import functions as F

overdrawn = bronze.filter(F.col("amount") > F.col("origin_balance_before"))
print(overdrawn.count())

In [0]:
display(overdrawn.groupBy("transaction_type").count().orderBy(F.desc("count")))

In [0]:
print(bronze.filter(F.col("origin_balance_before") == 0).count())